# Dynamic Factor Model (Turkey, Pipeline B)

**Model**: DFM via `nowcastDFM::dfm`  
**Target**: `ngdprsaxdctrq`  
**Variables**: Validation-selected Cat2 monthly set (4 predictors) + target = 5 total. COVID dummies, Tier-C short-history variables, and broader Cat3 variables excluded from the final specification.  
**Train**: 1995-01-01 to 2011-12-31 / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: YES (factor loadings depend on variance; EM algorithm standardizes internally)  
**HP tuning**: YES — feature panel selected on 2012-2017 validation RMSFE across m1/m2/m3; final DFM retrained through 2017.  
**Estimation**: EM algorithm, one fit (not rolling). Kalman filter handles ragged-edge nowcasting for the Cat3 monthly panel.  
**Prediction window**: 30-year rolling window (360 months) passed to predict_dfm (memory bound).  
**Deviation from US DFM**: Turkey has only 2 quarterly vars (ngdprsaxdctrq + real_gdp_i). US DFM excluded all  
quarterly Cat3 because nQ=23 caused eye((5*23)^2)=115x115 OOM. With nQ=2, eye(100) is trivial — both quarterly  
Cat2 was selected over selected10 and Cat3 on the pre-test validation window, then frozen for 2018-2025 evaluation.  
**COVID dummies excluded**: Zero variance in 1995-2011 training window crashes set_prior() / EM init.  
Same treatment as US DFM (Lenza-Primiceri 2022 defense: dummies used where variance is non-zero).


In [1]:
library(tidyverse)
library(nowcastDFM)

source("../data/helpers.R")

metadata <- read_csv("../turkey_data/meta_data_tr.csv", show_col_types = FALSE)

# All Cat3 vars (22) — monthly only (COVID excluded: zero variance in training)
cat3_features <- c(
  "altin_rezerv_var", "auto_prod", "bist100", "consu_i", "cpi_sa",
  "deposit_i", "doviz_rezerv_var", "emp_rate", "empl_num", "fin_acc",
  "ipi_sa", "m3", "maden_ciro_endeksi_sa", "ppi", "reer",
  "resmi_rezerv_var", "tax", "total_prod", "tourist", "unemp_num",
  "unemp_rate", "usd_try_avg"
)

# Cat2 was selected by 2012-2017 validation RMSFE across m1/m2/m3.
cat2_features <- c("ipi_sa", "usd_try_avg", "cpi_sa", "fin_acc")

# All Tier-C vars (32) — short history, DFM-only; Kalman filter handles sparse starts
# real_gdp_i is quarterly (freq=q) — included because nQ=2 total (trivial Kalman size)
tier_c_features <- c(
  "1week-repo", "auto_exp_vol_i", "auto_imp_vol_i", "bus_conf",
  "card_pmt_i", "card_pmt_i_r", "card_trans", "commprop_sales_sa",
  "cons_conf", "cur_sa", "dg_exp_vol_i", "dg_imp_vol_i",
  "elec_cons", "elec_prod", "exp_ind_auto", "exp_ind_dg",
  "exp_vol_i", "hh_pmt", "hh_pmt_r", "house_sales_sa",
  "household_credits", "imp_ind_auto", "imp_ind_dg", "imp_vol_i",
  "is_ekon_ciro_sa", "loans", "non_fin_firms_credits", "real_gdp_i",
  "retail_sales_nsa", "retail_sales_sa", "sanayi_ins_tic_ciro_sa",
  "total_prop_sales"
)

dfm_features <- cat2_features

data <- read_csv("../turkey_data/data_tf_monthly_tr.csv", show_col_types = FALSE) %>%
    arrange(date)

target_variable  <- "ngdprsaxdctrq"
train_start_date <- "1995-01-01"
test_start_date  <- "2018-03-01"
test_end_date    <- "2025-12-01"

# End-of-quarter dates (Mar/Jun/Sep/Dec) for test period
test_dates <- seq(as.Date("2018-03-01"), as.Date(test_end_date), by = "3 months")

test <- data %>%
    dplyr::filter(date >= as.Date(train_start_date), date <= as.Date(test_end_date)) %>%
    data.frame()

for (col in colnames(test)) {
    if (is.numeric(test[, col]) && sum(is.infinite(test[, col])) > 0) {
        test[is.infinite(test[, col]), col] <- NA
    }
}

cat("DFM Turkey: target =", target_variable, "| Cat3 =", length(cat3_features),
    "| Tier-C available =", length(tier_c_features),
    "| Used =", length(dfm_features) + 1, "vars (validation-selected Cat2 + target; Tier-C excluded)\n")


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     


── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


Zorunlu paket yükleniyor: matlab




Attaching package: 'matlab'




The following object is masked from 'package:stats':

    reshape




...


Attaching package: 'signal'




The following objects are masked from 'package:pracma':

    conv, ifft, interp1, pchip, polyval, roots




The following object is masked from 'package:dplyr':

    filter




The following objects are masked from 'package:stats':

    filter, poly




DFM Turkey: target = ngdprsaxdctrq | Cat3 = 22 | Tier-C available = 32 | Used = 5 vars (validation-selected Cat2 + target; Tier-C excluded)


In [2]:
# Patch nowcastDFM:::init_conds — cov(res[, idx_iM]) fails when a block has exactly
# 1 monthly variable because R drops dimensions, returning a vector instead of a matrix.
# Fix: wrap with as.matrix(..., drop=FALSE) to preserve 2-D shape.
# Turkey block_l has 4 vars, block_s has 4 — unlikely to trigger, included for safety.
local({
  fn <- nowcastDFM:::init_conds
  b  <- body(fn)
  replace_cov <- function(node) {
    if (is.call(node)) {
      txt <- deparse(node)
      if (identical(txt, "cov(res[, idx_iM])")) {
        return(quote(cov(as.matrix(res[, idx_iM, drop = FALSE]))))
      }
      return(as.call(lapply(node, replace_cov)))
    }
    node
  }
  body(fn) <- replace_cov(b)
  assignInNamespace("init_conds", fn, ns = "nowcastDFM")
  cat("nowcastDFM:::init_conds patched (drop=FALSE for single-variable blocks)\n")
})

# Patch dfm to cap A eigenvalues to prevent explosion with highly missing data
local({
  fn <- nowcastDFM::dfm; b <- body(fn)
  replace_A <- function(node) {
    if (is.call(node)) {
      txt <- deparse(node)[1]
      if (grepl("A <- em_output\\$A_new", txt)) {
        return(quote({
          A <- em_output$A_new
          diag_A <- diag(A)
          diag_A <- pmin(pmax(diag_A, -0.95), 0.95)
          diag(A) <- diag_A
        }))
      }
      return(as.call(lapply(node, replace_A)))
    }
    node
  }
  body(fn) <- replace_A(b)
  assignInNamespace("dfm", fn, ns="nowcastDFM")
})


# Build DFM variable list: preserves column order from data_tf_monthly_tr.csv
col_names_dfm <- colnames(test)[2:ncol(test)]
col_names_dfm <- col_names_dfm[col_names_dfm %in% c(target_variable, dfm_features)]
col_names_dfm <- c(col_names_dfm[col_names_dfm != target_variable], target_variable)

# Build blocks from metadata (Bok et al. 2018 4-block structure)
blocks <- metadata %>%
    dplyr::filter(series %in% col_names_dfm) %>%
    dplyr::filter(series %in% colnames(test))
blocks <- blocks[match(col_names_dfm, blocks$series), ]
blocks <- blocks %>%
    select(starts_with("block_")) %>%
    select_if(~ sum(.) > 0) %>%
    data.frame()
print(paste("Blocks:", ncol(blocks), "block columns for", nrow(blocks), "variables"))
print(paste("Variables matched:", length(col_names_dfm)))

# Fit DFM once on pre-test data (1995-2017). The Cat2 panel was selected
# using 2012-2017 validation performance, then frozen for 2018-2025 testing.
train_cols <- c("date", col_names_dfm)
train <- test %>%
    dplyr::filter(date <= as.Date("2017-12-31")) %>%
    dplyr::filter(date >= as.Date(train_start_date)) %>%
    data.frame()
train <- train[, train_cols]
train$date <- as.character(train$date)   # dfm() requires character dates
cat("Fitting DFM on", nrow(train), "months x", nrow(blocks), "variables...\n")
output_dfm <- dfm(data = train, blocks = blocks, max_iter = 500)
cat("DFM fitted\n")


nowcastDFM:::init_conds patched (drop=FALSE for single-variable blocks)


[1] "Blocks: 2 block columns for 5 variables"


[1] "Variables matched: 5"


Fitting DFM on 276 months x 5 variables...


Now running iteration number 20 out of a maximum of 500



...

Now running iteration number 80 out of a maximum of 500



Loglik: -394.4651; % change: -0.0039%



Now running iteration number 100 out of a maximum of 500



Loglik: -394.3095; % change: -0.0015%



DFM fitted


In [3]:
# Rolling prediction loop
vintage_offsets <- c(m1 = -2, m2 = -1, m3 = 0, post1 = 1, post2 = 2)
pred_dict <- data.frame(date = test_dates)
for (lag_name in names(vintage_offsets)) {
    pred_dict[, lag_name] <- NA
}

# Cap window to 30 years (360 months) to bound Kalman smoother memory.
# Turkey training starts 1995; test starts 2018 (~276 months). No trimming needed
# in early quarters, but included for robustness as test progresses toward 2025.
WINDOW_MONTHS <- 360

for (i in 1:length(test_dates)) {
    for (lag_name in names(vintage_offsets)) {
        vintage_date <- shift_month(test_dates[i], vintage_offsets[[lag_name]])
        lagged_data <- gen_vintage_data(metadata, test, test_dates[i], vintage_date)
        lagged_data <- data.frame(lagged_data)
        lagged_data <- lagged_data[, train_cols]  # match DFM training columns
        # Keep date as Date (NOT character): predict_dfm's add_month loop assigns as.Date()
        # back to the date column; if column is character, Date coerces to integer ("17258")
        # causing substr() to return "" -> month=NA -> if(month==12) crashes.
        lagged_data[lagged_data$date == test_dates[i], target_variable] <- NA

        # Trim to rolling window
        if (nrow(lagged_data) > WINDOW_MONTHS) {
            lagged_data <- tail(lagged_data, WINDOW_MONTHS)
        }

        prediction <- tryCatch({
            predict_dfm(lagged_data, output_dfm) %>%
                dplyr::filter(date == test_dates[i]) %>%
                select(!!target_variable) %>%
                pull()
        }, error = function(e) {
            cat("ERR", i, lag_name, conditionMessage(e), "\n")
            NA
        })
        pred_dict[pred_dict$date == test_dates[i], lag_name] <- prediction
    }
    if (i %% 8 == 0) print(paste(i, "/", length(test_dates)))
}

# Extract actuals at quarter-end months matching test_dates
actuals <- test %>%
    dplyr::filter(date %in% as.Date(test_dates)) %>%
    select(!!target_variable) %>%
    pull()


[1] "8 / 32"
[1] "16 / 32"
[1] "24 / 32"
[1] "32 / 32"


In [4]:
# Save predictions
dir.create("../turkey_predictions", showWarnings = FALSE)
for (lag_name in names(vintage_offsets)) {
    df_out <- data.frame(
        date = test_dates,
        actual = actuals,
        prediction = pred_dict[, lag_name]
    )
    write.csv(df_out, paste0("../turkey_predictions/dfm_tr_", lag_name, ".csv"), row.names = FALSE)
    cat("Saved dfm_tr_", lag_name, ".csv (\n", nrow(df_out), " rows)\n", sep="")
}

# Evaluation
panels <- list(
    "Pre-crisis (2018-2019)" = c("2018-03-01", "2019-12-31"),
    "COVID      (2020-2021)" = c("2020-06-01", "2021-12-31"),
    "Post-COVID (2022-2025)" = c("2022-03-01", "2025-12-31"),
    "Full test  (2018-2025)" = c("2018-03-01", "2025-12-31")
)

rmse <- function(a, p) sqrt(mean((a - p)^2, na.rm = TRUE))
mae  <- function(a, p) mean(abs(a - p), na.rm = TRUE)

cat("DFM (Turkey) — Evaluation by Panel & Vintage\n")
for (pn in names(panels)) {
    rng <- panels[[pn]]
    m <- test_dates >= rng[1] & test_dates <= rng[2]
    cat("--- ", pn, " ---\n")
    for (lag_name in names(vintage_offsets)) {
        cat("  ", lag_name, "  RMSFE=",
            format(rmse(actuals[m], pred_dict[m, lag_name]), digits = 6),
            "  MAE=",
            format(mae(actuals[m], pred_dict[m, lag_name]), digits = 6), "\n")
    }
}


Saved dfm_tr_m1.csv (
32 rows)
Saved dfm_tr_m2.csv (
32 rows)
Saved dfm_tr_m3.csv (
32 rows)
Saved dfm_tr_post1.csv (
32 rows)
Saved dfm_tr_post2.csv (
32 rows)


DFM (Turkey) — Evaluation by Panel & Vintage


---  Pre-crisis (2018-2019)  ---
   m1   RMSFE= 0.0113463   MAE= 0.00999068 
   m2   RMSFE= 0.00870383   MAE= 0.00609206 
   m3   RMSFE= 0.00947663   MAE= 0.00819165 
   post1   RMSFE= 0.00517959   MAE= 0.00447764 
   post2   RMSFE= 0.00782291   MAE= 0.00656738 
---  COVID      (2020-2021)  ---
   m1   RMSFE= 0.0666622   MAE= 0.0434104 
   m2   RMSFE= 0.0441155   MAE= 0.0292426 
   m3   RMSFE= 0.0454866   MAE= 0.0284345 
   post1   RMSFE= 0.0370872   MAE= 0.0260533 
   post2   RMSFE= 0.0275432   MAE= 0.0212646 
---  Post-COVID (2022-2025)  ---
   m1   RMSFE= 0.0179249   MAE= 0.0126161 
   m2   RMSFE= 0.0182753   MAE= 0.0129953 
   m3   RMSFE= 0.0237231   MAE= 0.0188608 
   post1   RMSFE= 0.0181632   MAE= 0.0146934 
   post2   RMSFE= 0.0203538   MAE= 0.0161583 
---  Full test  (2018-2025)  ---
   m1   RMSFE= 0.0341438   MAE= 0.0184665 
   m2   RMSFE= 0.0248112   MAE= 0.0147682 
   m3   RMSFE= 0.0275635   MAE= 0.0180194 
   post1   RMSFE= 0.0221002   MAE= 0.0148697 
   post2   RMSFE= 0.0